In [0]:
display(dbutils.fs.ls("/Volumes/finance_domain_project/default/raw_data"))



In [0]:
loans_raw_df= spark.read \
.format("csv") \
.option("header", True) \
.option("inferSchema",True) \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/loans_data_csv")


In [0]:
loans_schema = 'loan_id string, member_id string, loan_amount float, funded_amount float,loan_term_months string,interest_rate float,monthly_installment float, issue_date string,loan_status string,loan_purpose string,loan_title string'

In [0]:
loans_raw_df= spark.read \
.format("csv") \
.option("header", True) \
.schema(loans_schema) \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/loans_data_csv")


In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
loans_df_ingest_date = loans_raw_df.withColumn("ingest_date",current_timestamp())

In [0]:
loans_df_ingest_date.createOrReplaceTempView("loans")

In [0]:
display(spark.sql("select count(*) from loans"))

In [0]:
display(spark.sql("select * from loans where loan_amount is null"))

In [0]:
columns_to_check= ["loan_amount","funded_amount","loan_term_months","interest_rate","monthly_installment","issue_date","loan_status","loan_purpose"]

In [0]:
loans_filtered_df = loans_df_ingest_date.na.drop(subset=columns_to_check)

In [0]:
loans_filtered_df.count()

In [0]:
loans_filtered_df.createOrReplaceTempView("loans")

In [0]:
display(spark.sql("select * from loans"))

In [0]:
from pyspark.sql.functions import regexp_replace, col 

In [0]:
loans_term_modified_df=loans_filtered_df.withColumn("loan_term_months",(regexp_replace(col("loan_term_months"),"months","").cast("int")/12).cast("int")).withColumnRenamed("loan_term_months","loan_term_years")

In [0]:
display(spark.sql("select loan_purpose, count(*) as total from loans group by loan_purpose order by total desc"))

In [0]:
loan_purpose_lookup=["debt_consolidation","credit_card","home_improvement","other","major_purchase","medical","small_business","car","vacation","moving","house","wedding","renewable_energy","educational"]

In [0]:
from pyspark.sql.functions import when

In [0]:
loans_purpose_modified = loans_term_modified_df.withColumn("loan_purpose",when(col("loan_purpose").isin(loan_purpose_lookup),col("loan_purpose")).otherwise("other"))

In [0]:
loans_purpose_modified.createOrReplaceTempView("loans")

In [0]:
display(spark.sql("select loan_purpose, count(*) as total from loans group by loan_purpose order by total desc"))

In [0]:
loans_purpose_modified.write\
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_csv")\
.save()


In [0]:
loans_purpose_modified.write\
.format("parquet") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_parquet")\
.save()
